# 06 - Logistic Regression & Multicollinearity

Fit a logistic regression on the WoE features selected in `05_feature_selection` (train only; the test set is reserved for evaluation in the next step).

**Protected attributes** (`CODE_GENDER`, `NAME_FAMILY_STATUS`) are excluded up front: anti-discrimination rules (e.g. ECOA, EU law) prohibit using sex and marital status in credit decisions.

1. **Baseline model**: coefficients, p-values and VIF with all remaining candidates.
2. **Stepwise cleanup**: refit and drop one feature at a time, by priority:
   positive coefficient → p-value > 0.05 → VIF > 5.
   Since WoE = ln(%good / %bad), every coefficient must be **negative**.
3. **Final model**: review coefficients and VIF, check open issues from `PROGRESS.md`.
4. Save the model and its artifacts to `outputs/`.

## 1. Setup

In [1]:
import pickle
import sys
from pathlib import Path

import pandas as pd

# Add project root to sys.path so `src` can be imported from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.model import coefficient_table, compute_vif, fit_logit, stepwise_drop

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MAX_P = 0.05
MAX_VIF = 5.0

# Excluded for fairness / regulatory reasons, regardless of predictive power
PROTECTED = ["CODE_GENDER", "NAME_FAMILY_STATUS"]

## 2. Load WoE-encoded train data

In [2]:
woe_train = pd.read_pickle(DATA_DIR / "woe_train.pkl")
iv = pd.read_csv(OUTPUT_DIR / "iv_summary.csv", index_col=0)

X_train = woe_train.drop(columns=["TARGET", *PROTECTED])
y_train = woe_train["TARGET"]
print(f"X_train: {X_train.shape} | default rate: {y_train.mean():.2%}")

X_train: (246008, 20) | default rate: 8.07%


## 3. Baseline model (all candidates except protected attributes)

Model diagnostics table: coefficient, p-value, VIF and IV side by side.

In [3]:
def diagnostics(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """Coefficients, p-values, VIF and IV of a fitted logit, sorted by IV."""
    table = coefficient_table(fit_logit(X, y)).join(compute_vif(X)).join(iv["iv"])
    table["violation"] = (
        (table["coef"] > 0).map({True: "positive_coef ", False: ""})
        + (table["p_value"] > MAX_P).map({True: "p_value ", False: ""})
        + (table["vif"] > MAX_VIF).map({True: "vif", False: ""})
    ).str.strip()
    return table.sort_values("iv", ascending=False)


baseline = diagnostics(X_train, y_train)
baseline.round(4)

,coef,std_err,p_value,vif,iv,violation
EXT_SOURCE_3,-0.8519,0.0138,0.0000,1.0643,0.3296,
EXT_SOURCE_2,-0.7460,0.0152,0.0000,1.2145,0.3030,
EXT_SOURCE_1,-0.5488,0.0219,0.0000,1.1370,0.1512,
YEARS_EMPLOYED,-0.3775,0.0303,0.0000,1.6734,0.1116,
AMT_GOODS_PRICE,-0.3296,0.0368,0.0000,1.0884,0.0913,
AGE_YEARS,0.1940,0.0357,0.0000,1.9845,0.0867,positive_coef
OCCUPATION_TYPE,-0.3118,0.0324,0.0000,1.3629,0.0754,
ORGANIZATION_TYPE,-0.3588,0.0460,0.0000,2.2979,0.0596,
NAME_INCOME_TYPE,-0.0147,0.0465,0.7518,2.1162,0.0558,p_value
REGION_RATING_CLIENT_W_CITY,-0.3319,0.0443,0.0000,1.8885,0.0516,


In [4]:
# Features violating at least one rule in the baseline model
baseline[baseline["violation"] != ""].round(4)

,coef,std_err,p_value,vif,iv,violation
AGE_YEARS,0.1940,0.0357,0.0000,1.9845,0.0867,positive_coef
NAME_INCOME_TYPE,-0.0147,0.0465,0.7518,2.1162,0.0558,p_value
REGION_POPULATION_RELATIVE,0.1556,0.0610,0.0108,1.8262,0.0342,positive_coef
REG_CITY_NOT_WORK_CITY,0.0740,0.0490,0.1307,1.4000,0.0330,positive_coef p_value


## 4. Stepwise cleanup

Drop one feature per step and refit, so a removal can fix issues caused by collinearity (e.g. a sign flip).

In [5]:
final_features, drop_log = stepwise_drop(X_train, y_train, max_p=MAX_P, max_vif=MAX_VIF)

print(f"{X_train.shape[1]} -> {len(final_features)} features")
drop_log.round(4)

20 -> 16 features


,step,feature,reason,value
0,1,AGE_YEARS,positive_coef,0.1940
1,2,REGION_POPULATION_RELATIVE,positive_coef,0.1558
2,3,REG_CITY_NOT_WORK_CITY,positive_coef,0.0780
3,4,NAME_INCOME_TYPE,positive_coef,0.0367


## 5. Final model

In [6]:
model = fit_logit(X_train[final_features], y_train)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                 TARGET   No. Observations:               246008
Model:                          Logit   Df Residuals:                   245991
Method:                           MLE   Df Model:                           16
Date:                Wed, 04 Nov 2026   Pseudo R-squ.:                  0.1002
Time:                        14:08:54   Log-Likelihood:                -62098.
converged:                       True   LL-Null:                       -69017.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const                          -2.4358      0.008   -299.588      0.000      -2.452      -2.420
EXT_SOURCE_3                   -0.8478      0.014    -61.688      0.000      -0.

In [7]:
final = diagnostics(X_train[final_features], y_train)
assert (final["coef"] < 0).all(), "All coefficients must be negative"
final.round(4)

,coef,std_err,p_value,vif,iv,violation
EXT_SOURCE_3,-0.8478,0.0137,0.0000,1.0591,0.3296,
EXT_SOURCE_2,-0.7386,0.0151,0.0000,1.2045,0.3030,
EXT_SOURCE_1,-0.5188,0.0212,0.0000,1.0940,0.1512,
YEARS_EMPLOYED,-0.3261,0.0286,0.0000,1.5193,0.1116,
AMT_GOODS_PRICE,-0.3098,0.0366,0.0000,1.0806,0.0913,
OCCUPATION_TYPE,-0.3127,0.0319,0.0000,1.3307,0.0754,
ORGANIZATION_TYPE,-0.3065,0.0406,0.0000,1.6361,0.0596,
REGION_RATING_CLIENT_W_CITY,-0.2752,0.0369,0.0000,1.1806,0.0516,
NAME_EDUCATION_TYPE,-0.5452,0.0393,0.0000,1.1737,0.0486,
YEARS_LAST_PHONE_CHANGE,-0.2104,0.0392,0.0000,1.0753,0.0447,


## 6. Review open issues

- `REG_CITY_NOT_LIVE_CITY`: IV close to 0.02 on train (below on test). Decision left to the model, not to the test set.
- `AGE_YEARS`: decent IV but its sign flips in the joint model (overlaps with `EXT_SOURCE_1`, `YEARS_EMPLOYED`) → dropped is accepted.
- Pensioner group encoded by several features (`NAME_INCOME_TYPE`, `ORGANIZATION_TYPE`, `YEARS_EMPLOYED`): check VIF.

In [8]:
watch = [
    "REG_CITY_NOT_LIVE_CITY",
    "AGE_YEARS",
    "NAME_INCOME_TYPE",
    "ORGANIZATION_TYPE",
    "YEARS_EMPLOYED",
]
status = baseline.reindex(watch)[["coef", "p_value", "vif", "iv"]]
status["final"] = [
    "kept" if f in final_features
    else "not a candidate" if f not in X_train.columns
    else "dropped: " + drop_log.set_index("feature").loc[f, "reason"]
    for f in watch
]
status.round(4)

,coef,p_value,vif,iv,final
REG_CITY_NOT_LIVE_CITY,-0.3176,0.0000,1.2658,0.0235,kept
AGE_YEARS,0.1940,0.0000,1.9845,0.0867,dropped: positive_coef
NAME_INCOME_TYPE,-0.0147,0.7518,2.1162,0.0558,dropped: positive_coef
ORGANIZATION_TYPE,-0.3588,0.0000,2.2979,0.0596,kept
YEARS_EMPLOYED,-0.3775,0.0000,1.6734,0.1116,kept


## 7. Save artifacts

In [9]:
# Strip training data from the result to keep the pickle small
model.remove_data()
with open(OUTPUT_DIR / "logit_model.pkl", "wb") as f:
    pickle.dump(model, f)

final.to_csv(OUTPUT_DIR / "model_coefficients.csv")
drop_log.to_csv(OUTPUT_DIR / "model_drop_log.csv", index=False)
pd.Series(final_features, name="feature").to_csv(OUTPUT_DIR / "model_features.csv", index=False)
print("Saved logit_model.pkl, model_coefficients.csv, model_drop_log.csv, model_features.csv")

Saved logit_model.pkl, model_coefficients.csv, model_drop_log.csv, model_features.csv
